In [ ]:
from pathlib import Path
import shutil
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation

from utils.setup_utils import create_minio_spark_session
import utils.cbrfc_wsv_esp_utils as cbrfc_wsv_esp_utils

### Setup eval

In [ ]:
# create spark session
spark = create_minio_spark_session()

In [ ]:
# define evaluation
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# configure download
ev.download.configure(
    api_key='your_key_here'
)

### Define new crosswalk and POR

In [ ]:
usgs_to_rfc_dict = {
    'usgs-09149500':'cbrfc-dlac2', # in TEEHR
    'usgs-09361500':'cbrfc-drgc2', # in TEEHR
    'usgs-09260050':'cbrfc-ydlc2', # in TEEHR
}



start_date = '2016-01-01 00:00:00'
end_date = '2022-11-30 00:00:00'

### Update domain tables ahead of timeseries ingests

In [ ]:
primary_locs = usgs_to_rfc_dict.keys()

# add configurations for cbrfc wsv's (use existing for usgs/nwm)
from teehr import Configuration
configuration = Configuration(
    name="cbrfc_forecast",
    timeseries_type="secondary",
    description="CBRFC forecasts",
)
ev.configurations.add(configuration)

# add unit for volume
from teehr import Unit
unit = Unit(
    name='m^3',
    long_name='Cubic Meter'
)
ev.units.add(unit)

# add variable for monthly wsv's
from teehr import Variable
variable = Variable(
    name="wsv_monthly_inst",
    long_name="Monthly instantaneous water supply volume"
)
ev.variables.add(variable)

# add cbrfc crosswalk to table
crosswalk_dict = {
    'primary_location_id': usgs_to_rfc_dict.keys(),
    'secondary_location_id':usgs_to_rfc_dict.values()
}
temp = pd.DataFrame(crosswalk_dict)
ev.location_crosswalks.load_dataframe(temp)

### Load USGS timeseries and NWM-retrospective from warehouse

In [ ]:
ev.download.primary_timeseries(
    primary_location_id=primary_locs,
    configuration_name="usgs_observations",
    variable_name="streamflow_none_inst",
    start_date=start_date,
    end_date=end_date,
    load=True,
)

In [ ]:
ev.download.secondary_timeseries(
    primary_location_id=primary_locs,
    configuration_name="nwm30_retrospective",
    variable_name="streamflow_hourly_inst",
    start_date=start_date,
    end_date=end_date,
    load=True,
)

### Calculate monthly WSV from USGS/NWM-retrospective

In [ ]:
def compute_monthly_wsv(sdf, constant_field_values, ts_type):
    from pyspark.sql import functions as F
    
    # get month and year
    sdf_with_month = sdf.withColumn("year", F.year(F.col("value_time"))) \
                      .withColumn("month", F.month(F.col("value_time")))
    
    # define uniqueness_fields
    if ts_type == 'primary':
        uniqueness_fields = ['year', 'month', 'location_id', 'reference_time']
    elif ts_type == 'secondary':
        uniqueness_fields = ['year', 'month', 'location_id', 'reference_time', 'member']
    else:
        raise ValueError(f"unspecified or invalid ts_type provided: {ts_type}")
    
    # get monthly total flow
    monthly_aggregates = sdf_with_month.groupBy(uniqueness_fields) \
                                       .agg(F.sum("value").alias("total_flow"))
    
    # convert total flow to volumes
    monthly_volume_sdf = monthly_aggregates.withColumn(
        "monthly_volume", 
        F.col("total_flow") * 3600
    )
    
    # add a value_time that corresponds to the first of the month
    monthly_volume_sdf = monthly_volume_sdf.withColumn(
        "first_of_month", 
        F.make_timestamp(F.col("year"), F.col("month"), F.lit(1), F.lit(0), F.lit(0), F.lit(0))
    )
    
    # drop year, month, total_flow
    dropped_cols = ['year', 'month', 'total_flow']
    formatted_sdf = monthly_volume_sdf.drop(*dropped_cols)
    
    # rename generated columns to fit schema
    formatted_sdf = formatted_sdf.withColumnsRenamed({"monthly_volume": "value", "first_of_month": "value_time"})
    
    # map the constant field values
    for key in constant_field_values.keys():
        formatted_sdf = formatted_sdf.withColumn(key, F.lit(constant_field_values[key]))

    return formatted_sdf

#### Add monthly WSVs for USGS to eval

In [ ]:
# define filters
formatted_locs = ",".join([f"'{x}'" for x in usgs_to_rfc_dict.keys()])

# query timeseries
sdf = ev.primary_timeseries.filter([
    f"location_id IN ({formatted_locs})",
    f"value_time >= '{start_date}'",
    f"value_time <= '{end_date}'"
]).to_sdf()

In [ ]:
# define the required constant_field_values
cfvs = {
    'configuration_name': 'usgs_observations',
    'variable_name': 'wsv_monthly_inst',
    'unit_name': 'm^3'
}

# obtain wsv_sdf
obs_wsv_sdf = compute_monthly_wsv(sdf=sdf, constant_field_values=cfvs, ts_type='primary')

In [ ]:
ev.primary_timeseries.load_dataframe(obs_wsv_sdf)

#### Add monthly WSVs for NWM-retrospective to eval

In [ ]:
# Get non-cbrfc crosswalks for the pilot locations
crosswalk_df = ev.location_crosswalks.to_pandas()
temp = crosswalk_df[crosswalk_df['primary_location_id'].isin(primary_locs)]
nwm30_ids = temp[temp['secondary_location_id'].str.startswith('nwm30')]
nwm30_ids

In [ ]:
formatted_locs = ",".join([f"'{x}'" for x in nwm30_ids['secondary_location_id']])

# query timeseries
sdf = ev.secondary_timeseries.filter([
    "configuration_name = 'nwm30_retrospective'",
    f"location_id IN ({formatted_locs})",
    f"value_time >= '{start_date}'",
    f"value_time <= '{end_date}'"
]).to_sdf()

In [ ]:
# define the required constant_field_values
cfvs = {
    'configuration_name': 'nwm30_retrospective',
    'variable_name': 'wsv_monthly_inst',
    'unit_name': 'm^3'
}

# obtain wsv_sdf
retrospective_wsv_sdf = compute_monthly_wsv(sdf=sdf, constant_field_values=cfvs, ts_type='secondary')

In [ ]:
ev.secondary_timeseries.load_dataframe(retrospective_wsv_sdf)

### Query cbrfc 32 month esp wsv's

In [ ]:
# wipe local file dir if exists
local_timeseries_dir = Path(Path.cwd(), 'local_timeseries')
if local_timeseries_dir.exists():
    shutil.rmtree(local_timeseries_dir)

In [ ]:
# get list of LIDs from crosswalk
lids = list(usgs_to_rfc_dict.values())
print(lids)

In [ ]:
# query the CBRFC archive for espmvol data for the LIDs in the crosswalk
ts_data = {}
for lid in lids:
    # format lid to for use in query
    formatted_lid = lid.split('-')[1].upper()

    # define constant field values
    constant_field_values = {
        'location_id': lid,
        'configuration_name': 'cbrfc_forecast',
        'variable_name': 'wsv_monthly_inst'
    }

    # query the timeseries data
    ts_data[lid] = cbrfc_wsv_esp_utils.query_espmvol_forecast(
        formatted_lid,
        years=[16, 17],
        constant_field_values=constant_field_values
        )

In [ ]:
# create local file directory
cbrfc_timeseries_dir = Path(local_timeseries_dir, 'cbrfc_esp_wsvs')
cbrfc_timeseries_dir.mkdir(exist_ok=True, parents=True)

for lid in lids:
    fname = f'{lid}_wsv.parquet'
    fpath = Path(cbrfc_timeseries_dir, fname)
    ts_data[lid].to_parquet(fpath)

### Load cbrfc 32 month esp wsv's

In [ ]:
# ingest the parquet files
ev.secondary_timeseries.load_parquet(
    in_path=cbrfc_timeseries_dir
)

### Kill spark

In [ ]:
spark.stop()